<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/U_S_Critical_Infrastructure_Dependency_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Build a National Infrastructure Digital Twin to simulate cascading failures across the 16 CISA critical infrastructure sectors. This involves mapping nodes (e.g., power plants, hospitals), establishing cross-sector dependencies, performing network vulnerability analysis (PageRank, K-Core), and simulating failure propagation under stress scenarios. The final output will include a resilience index report and an interactive dashboard.

## Environment Setup and Data Pipeline Initialization

### Subtask:
Install necessary libraries and initialize the multi-layer infrastructure graph structure.


**Reasoning**:
First, I need to install the specialized libraries required for geospatial and network analysis.



In [1]:
!pip install networkx geopandas osmnx h3 pydeck plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.3 MB/s eta 0:00:00


**Reasoning**:
I will initialize a NetworkX MultiDiGraph and define the constants for the 16 CISA critical infrastructure sectors to serve as the foundation for the digital twin.



In [2]:
import networkx as nx
import pandas as pd
import numpy as np

# Define the 16 CISA Critical Infrastructure Sectors
CISA_SECTORS = [
    'Chemical', 'Commercial Facilities', 'Communications', 'Critical Manufacturing',
    'Dams', 'Defense Industrial Base', 'Emergency Services', 'Energy',
    'Financial Services', 'Food and Agriculture', 'Government Facilities',
    'Healthcare and Public Health', 'Information Technology', 'Nuclear Reactors, Materials, and Waste',
    'Transportation Systems', 'Water and Wastewater Systems'
]

# Initialize the MultiDiGraph for the Digital Twin
# MultiDiGraph allows multiple edges between nodes, which is essential for different types of dependencies
infrastructure_graph = nx.MultiDiGraph()

print(f'Digital Twin Graph initialized.')
print(f'Targeting {len(CISA_SECTORS)} CISA Sectors for simulation.')

Digital Twin Graph initialized.
Targeting 16 CISA Sectors for simulation.


## Geospatial Data Acquisition and Node Mapping

### Subtask:
Generate and map infrastructure nodes for the 16 CISA sectors with geographic coordinates.


**Reasoning**:
I will create a data generation script that populates our digital twin with infrastructure nodes across the 16 sectors, providing them with geographic coordinates and unique attributes.



In [3]:
import random

def generate_infrastructure_nodes(sectors, n_per_sector=10, bbox=[-122.5, 37.7, -122.3, 37.8]):
    nodes = []
    node_id_counter = 0
    for sector in sectors:
        for _ in range(n_per_sector):
            node_id = f"{sector.replace(' ', '_')}_{node_id_counter}"
            lon = random.uniform(bbox[0], bbox[2])
            lat = random.uniform(bbox[1], bbox[3])
            criticality = round(random.uniform(0.5, 1.0), 2)
            nodes.append({
                'node_id': node_id,
                'sector': sector,
                'pos': (lon, lat),
                'criticality': criticality,
                'status': 'operational'
            })
            node_id_counter += 1
    return nodes

# Generate nodes
infrastructure_nodes = generate_infrastructure_nodes(CISA_SECTORS)

# Add to graph
for node in infrastructure_nodes:
    infrastructure_graph.add_node(node['node_id'], **node)

# Convert to GeoDataFrame for visualization/analysis
import geopandas as gpd
from shapely.geometry import Point

df_nodes = pd.DataFrame(infrastructure_nodes)
df_nodes['geometry'] = df_nodes['pos'].apply(lambda p: Point(p[0], p[1]))
gdf_nodes = gpd.GeoDataFrame(df_nodes, geometry='geometry', crs='EPSG:4326')

print(f"Total nodes added: {len(infrastructure_graph.nodes())}")
print(gdf_nodes.groupby('sector').size())

Total nodes added: 160
sector
Chemical                                  10
Commercial Facilities                     10
Communications                            10
Critical Manufacturing                    10
Dams                                      10
Defense Industrial Base                   10
Emergency Services                        10
Energy                                    10
Financial Services                        10
Food and Agriculture                      10
Government Facilities                     10
Healthcare and Public Health              10
Information Technology                    10
Nuclear Reactors, Materials, and Waste    10
Transportation Systems                    10
Water and Wastewater Systems              10
dtype: int64


## Build Multi-Layer Dependency Network

### Subtask:
Establish logical and spatial dependencies between sectors and implement H3 hexagonal binning.


**Reasoning**:
I will define the dependency rules between sectors and create the network edges in the MultiDiGraph. I'll also add H3 indexing to each node for spatial aggregation.



In [4]:
import h3

# Define logical dependencies: Sector A depends on Sector B
# Key = Dependent Sector, Value = Provider Sectors
DEPENDENCY_MAP = {
    'Water and Wastewater Systems': ['Energy', 'Communications'],
    'Communications': ['Energy', 'Information Technology'],
    'Healthcare and Public Health': ['Energy', 'Water and Wastewater Systems', 'Communications'],
    'Financial Services': ['Communications', 'Information Technology', 'Energy'],
    'Emergency Services': ['Transportation Systems', 'Communications', 'Energy'],
    'Energy': ['Information Technology', 'Critical Manufacturing']
}

# Add H3 indices to nodes for spatial analysis (Resolution 9)
H3_RES = 9
gdf_nodes['h3_index'] = gdf_nodes.apply(lambda row: h3.latlng_to_h3(row['pos'][1], row['pos'][0], H3_RES), axis=1)

# Update graph nodes with H3 index
for idx, row in gdf_nodes.iterrows():
    infrastructure_graph.nodes[row['node_id']]['h3_index'] = row['h3_index']

# Establish logical dependencies based on Sector proximity and rules
# For simplicity, we link nodes in a dependent sector to nodes in provider sectors within the same region
edge_count = 0
for dep_sector, providers in DEPENDENCY_MAP.items():
    dep_nodes = gdf_nodes[gdf_nodes['sector'] == dep_sector]['node_id'].tolist()
    for provider_sector in providers:
        prov_nodes = gdf_nodes[gdf_nodes['sector'] == provider_sector]['node_id'].tolist()
        # Create a few random dependency links for simulation purposes
        for d_node in dep_nodes:
            # Link each dependent node to 1-2 random provider nodes in that sector
            targets = random.sample(prov_nodes, k=random.randint(1, 2))
            for t_node in targets:
                infrastructure_graph.add_edge(t_node, d_node, type='dependency', weight=0.8)
                edge_count += 1

print(f"Dependency edges created: {edge_count}")
print(f"Total edges in graph: {infrastructure_graph.number_of_edges()}")
print("H3 binning completed at resolution 9.")

AttributeError: module 'h3' has no attribute 'latlng_to_h3'

**Reasoning**:
Correcting the H3 method name to 'latlng_to_cell' to align with the installed version 4.5.0 of the h3 library and completing the dependency edge creation.



In [5]:
import h3
import random

# Define logical dependencies: Sector A depends on Sector B
# Key = Dependent Sector, Value = Provider Sectors
DEPENDENCY_MAP = {
    'Water and Wastewater Systems': ['Energy', 'Communications'],
    'Communications': ['Energy', 'Information Technology'],
    'Healthcare and Public Health': ['Energy', 'Water and Wastewater Systems', 'Communications'],
    'Financial Services': ['Communications', 'Information Technology', 'Energy'],
    'Emergency Services': ['Transportation Systems', 'Communications', 'Energy'],
    'Energy': ['Information Technology', 'Critical Manufacturing']
}

# Add H3 indices to nodes using the updated H3-v4 API method: latlng_to_cell
H3_RES = 9
gdf_nodes['h3_index'] = gdf_nodes.apply(lambda row: h3.latlng_to_cell(row['pos'][1], row['pos'][0], H3_RES), axis=1)

# Update graph nodes with H3 index
for idx, row in gdf_nodes.iterrows():
    infrastructure_graph.nodes[row['node_id']]['h3_index'] = row['h3_index']

# Establish logical dependencies based on Sector proximity and rules
edge_count = 0
for dep_sector, providers in DEPENDENCY_MAP.items():
    dep_nodes = gdf_nodes[gdf_nodes['sector'] == dep_sector]['node_id'].tolist()
    for provider_sector in providers:
        prov_nodes = gdf_nodes[gdf_nodes['sector'] == provider_sector]['node_id'].tolist()
        # Create dependency links
        for d_node in dep_nodes:
            # Link each dependent node to 1-2 random provider nodes in that sector
            targets = random.sample(prov_nodes, k=random.randint(1, 2))
            for t_node in targets:
                infrastructure_graph.add_edge(t_node, d_node, type='dependency', weight=0.8)
                edge_count += 1

print(f"Dependency edges created: {edge_count}")
print(f"Total edges in graph: {infrastructure_graph.number_of_edges()}")
print("H3 binning completed successfully using latlng_to_cell.")

Dependency edges created: 228
Total edges in graph: 228
H3 binning completed successfully using latlng_to_cell.


## Network Centrality and Vulnerability Analysis

### Subtask:
Analyze the infrastructure graph to identify the most critical and vulnerable nodes using network science metrics.


**Reasoning**:
I will calculate PageRank, K-Core, and Betweenness Centrality for the infrastructure graph to quantify node importance and identify systemic vulnerabilities.



In [6]:
import networkx as nx

# 1. PageRank Calculation (Systemic Importance)
# We treat the dependency graph as a directed network where edges represent support/flow
pagerank_scores = nx.pagerank(infrastructure_graph, weight='weight')

# 2. Betweenness Centrality (Bridge Nodes)
betweenness_scores = nx.betweenness_centrality(infrastructure_graph)

# 3. K-Core Decomposition (Structural Core)
# K-Core requires a simple graph (no self-loops or multiple edges), so we use a directed graph version
g_simple = nx.DiGraph(infrastructure_graph)
k_core_scores = nx.core_number(g_simple)

# Map scores back to the GeoDataFrame
gdf_nodes['pagerank'] = gdf_nodes['node_id'].map(pagerank_scores)
gdf_nodes['betweenness'] = gdf_nodes['node_id'].map(betweenness_scores)
gdf_nodes['k_core'] = gdf_nodes['node_id'].map(k_core_scores)

# Calculate a composite 'Resilience Risk' score (Lower PageRank/K-Core + High Betweenness might indicate vulnerability)
# For now, let's identify the top 5 'Crown Jewel' nodes by PageRank
crown_jewels = gdf_nodes.nlargest(5, 'pagerank')[['node_id', 'sector', 'pagerank', 'k_core']]

print("Network Centrality Analysis Complete.")
print("\nTop 5 'Crown Jewel' Infrastructure Nodes (by PageRank):")
print(crown_jewels)

Network Centrality Analysis Complete.

Top 5 'Crown Jewel' Infrastructure Nodes (by PageRank):
                              node_id                        sector  pagerank  \
116  Healthcare_and_Public_Health_116  Healthcare and Public Health  0.018517   
113  Healthcare_and_Public_Health_113  Healthcare and Public Health  0.016448   
112  Healthcare_and_Public_Health_112  Healthcare and Public Health  0.016185   
119  Healthcare_and_Public_Health_119  Healthcare and Public Health  0.016007   
65              Emergency_Services_65            Emergency Services  0.013869   

     k_core  
116       4  
113       4  
112       4  
119       4  
65        4  


## Cascading Failure Simulation Engine

### Subtask:
Develop a percolation-based simulation engine to model failure propagation under stress scenarios.


**Reasoning**:
I will implement the cascading failure logic by defining a simulator that evaluates node status based on the health of its dependencies in the MultiDiGraph.



In [7]:
import copy

def simulate_cascade(graph, initial_failed_nodes, threshold=0.5):
    # Initialize node statuses
    status = {node: 'operational' for node in graph.nodes()}
    for node in initial_failed_nodes:
        status[node] = 'failed'

    active_nodes = set(initial_failed_nodes)
    cascade_history = []
    step = 0

    while active_nodes:
        newly_failed = []
        step += 1

        # Check each operational node to see if its dependencies are met
        for node in graph.nodes():
            if status[node] == 'operational':
                # Get predecessors (providers)
                providers = list(graph.predecessors(node))
                if not providers:
                    continue

                # Calculate failure ratio
                failed_providers = [p for p in providers if status[p] == 'failed']
                failure_ratio = len(failed_providers) / len(providers)

                if failure_ratio >= threshold:
                    newly_failed.append(node)

        # Update statuses for the next iteration
        for node in newly_failed:
            status[node] = 'failed'

        active_nodes = set(newly_failed)

        # Record snapshot
        snapshot = {s: list(status.values()).count(s) for s in ['operational', 'failed']}
        snapshot['step'] = step
        cascade_history.append(snapshot)

        if step > 20: break # Safety break

    return status, pd.DataFrame(cascade_history)

# Scenario: Targeting the top 'Crown Jewel' identified in the last step
target_node = crown_jewels.iloc[0]['node_id']
print(f'Simulating attack on: {target_node}')

final_status, df_cascade_results = simulate_cascade(infrastructure_graph, [target_node], threshold=0.3)

# Map final status back to a summary
gdf_nodes['final_status'] = gdf_nodes['node_id'].map(final_status)
impact_summary = gdf_nodes[gdf_nodes['final_status'] == 'failed'].groupby('sector').size().reset_index(name='failed_count')

print('\nSimulation Results:')
print(df_cascade_results)
print('\nImpact by Sector:')
print(impact_summary)

Simulating attack on: Healthcare_and_Public_Health_116

Simulation Results:
   operational  failed  step
0          159       1     1

Impact by Sector:
                         sector  failed_count
0  Healthcare and Public Health             1


## Interactive Digital Twin Dashboard

### Subtask:
Create an interactive visualization using Plotly and PyDeck to explore the infrastructure network, risk levels, and simulation results.


**Reasoning**:
I will start by creating the geospatial visualization using PyDeck. This will include a ScatterplotLayer for individual nodes and a ScreenGridLayer or H3HexagonLayer to visualize the spatial density and risk (PageRank) across the bounding box.



In [8]:
import pydeck as pdk

# Prepare data for PyDeck: Extract lat/lng from the position tuple
gdf_nodes['lng'] = gdf_nodes['pos'].apply(lambda x: x[0])
gdf_nodes['lat'] = gdf_nodes['pos'].apply(lambda x: x[1])

# Define a layer for the nodes
node_layer = pdk.Layer(
    'ScatterplotLayer',
    gdf_nodes,
    get_position=['lng', 'lat'],
    get_color='[200, 30, 0, 160]' if 'final_status' == 'failed' else '[30, 150, 30, 160]',
    get_radius=40,
    pickable=True,
    auto_highlight=True,
)

# Define a layer for the H3 Hexagons (Risk Heatmap based on PageRank)
# We aggregate PageRank by H3 cell for the visualization
h3_data = gdf_nodes.groupby('h3_index')['pagerank'].sum().reset_index()

h3_layer = pdk.Layer(
    'H3HexagonLayer',
    h3_data,
    get_hexagon='h3_index',
    get_fill_color='[pagerank * 1000, 50, 150, 180]',
    get_elevation='pagerank * 1000',
    elevation_scale=10,
    pickable=True,
    extruded=True,
)

# Set the viewport
view_state = pdk.ViewState(
    longitude=-122.4,
    latitude=37.75,
    zoom=11,
    pitch=45,
)

# Render the map
r = pdk.Deck(layers=[h3_layer, node_layer], initial_view_state=view_state, tooltip={'text': 'Sector: {sector}\nNode: {node_id}\nPageRank: {pagerank}'})
r.to_html('infrastructure_twin_map.html', notebook_display=True)
print('Interactive geospatial map generated.')

<IPython.core.display.Javascript object>

Interactive geospatial map generated.


**Reasoning**:
I will create Plotly charts to visualize the cascading failure results: a line chart for the failure progression and a bar chart for the impact summary by sector. This complements the PyDeck map for a full interactive dashboard experience.



In [9]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Failure Progression Over Time
fig_line = px.line(
    df_cascade_results,
    x='step',
    y=['operational', 'failed'],
    title='Infrastructure Cascading Failure Progression',
    labels={'value': 'Node Count', 'step': 'Simulation Step'},
    markers=True
)
fig_line.show()

# 2. Impact by Sector
fig_bar = px.bar(
    impact_summary,
    x='sector',
    y='failed_count',
    title='Total Node Failures by CISA Sector',
    color='failed_count',
    color_continuous_scale='Reds',
    labels={'failed_count': 'Number of Failed Nodes', 'sector': 'Infrastructure Sector'}
)
fig_bar.update_layout(xaxis_tickangle=-45)
fig_bar.show()

print('Plotly dashboard components generated successfully.')

Plotly dashboard components generated successfully.


## Final Task

### Subtask:
Generate the final deliverables including the .graphml network files, the risk_scores.csv, and a summary report of the U.S. Infrastructure Resilience Index.


**Reasoning**:
I will now export the graph data, save the risk scores to a CSV, and generate the final textual summary report to complete the project.



In [10]:
import networkx as nx
import pandas as pd

# 1. Export Graph to GraphML
# We need to convert the 'pos' tuple and other non-standard types to strings for GraphML compatibility
g_export = infrastructure_graph.copy()
for n, d in g_export.nodes(data=True):
    if 'pos' in d: d['pos'] = str(d['pos'])
    if 'geometry' in d: del d['geometry']

nx.write_graphml(g_export, 'infrastructure_twin.graphml')

# 2. Export Risk Scores to CSV
risk_report_df = gdf_nodes[['node_id', 'sector', 'criticality', 'h3_index', 'pagerank', 'betweenness', 'k_core', 'final_status']]
risk_report_df.to_csv('risk_scores.csv', index=False)

# 3. Calculate Global Resilience Index
total_nodes = len(gdf_nodes)
failed_nodes = len(gdf_nodes[gdf_nodes['final_status'] == 'failed'])
resilience_index = (total_nodes - failed_nodes) / total_nodes

# 4. Final Summary Report
print("--- NATIONAL INFRASTRUCTURE DIGITAL TWIN SUMMARY REPORT ---")
print(f"Total Infrastructure Nodes: {total_nodes}")
print(f"Total Sector Dependencies: {infrastructure_graph.number_of_edges()}")
print(f"Global Resilience Index: {resilience_index:.2f}")
print("\nTop 3 Systemically Critical Nodes (Crown Jewels):")
print(gdf_nodes.nlargest(3, 'pagerank')[['node_id', 'sector', 'pagerank']])
print("\nFiles Generated:")
print("- infrastructure_twin.graphml (Network Structure)")
print("- risk_scores.csv (Detailed Vulnerability Metrics)")
print("- infrastructure_twin_map.html (Geospatial Dashboard)")
print("----------------------------------------------------------")

--- NATIONAL INFRASTRUCTURE DIGITAL TWIN SUMMARY REPORT ---
Total Infrastructure Nodes: 160
Total Sector Dependencies: 228
Global Resilience Index: 0.99

Top 3 Systemically Critical Nodes (Crown Jewels):
                              node_id                        sector  pagerank
116  Healthcare_and_Public_Health_116  Healthcare and Public Health  0.018517
113  Healthcare_and_Public_Health_113  Healthcare and Public Health  0.016448
112  Healthcare_and_Public_Health_112  Healthcare and Public Health  0.016185

Files Generated:
- infrastructure_twin.graphml (Network Structure)
- risk_scores.csv (Detailed Vulnerability Metrics)
- infrastructure_twin_map.html (Geospatial Dashboard)
----------------------------------------------------------
